# Using POUNCE from Pyomo

[Pyomo](https://www.pyomo.org/) is an algebraic modeling language for Python. The **`pyomo-pounce`** plugin registers POUNCE with Pyomo's `SolverFactory`, so any Pyomo nonlinear program can be handed to POUNCE.

POUNCE speaks the AMPL NL/SOL protocol, so Pyomo drives it through the AMPL Solver Library (ASL) interface — exactly the way Pyomo drives IPOPT. Switching solvers is a one-line change: `SolverFactory('ipopt')` becomes `SolverFactory('pounce')`.

**Prerequisites**

* `pip install pyomo-pounce` (which also pulls in `pyomo`)
* the `pounce` binary on your `PATH` — build it with `cargo build --release --bin pounce` and add `target/release` to `PATH`

This notebook covers an unconstrained warm-up, the constrained HS071 problem, dual values, solver options, watching the solve, and reading the result object.

In [1]:
import pyomo_pounce  # noqa: F401 -- importing registers 'pounce' with SolverFactory
from pyomo.environ import (
    ConcreteModel,
    Constraint,
    Objective,
    RangeSet,
    SolverFactory,
    Suffix,
    Var,
    minimize,
    value,
)

solver = SolverFactory("pounce")
print("pounce available:", solver.available(exception_flag=False))

pounce available: True


## 1. An unconstrained problem

Start with a one-variable least-squares fit:

$$\min_{x} \; (x - 2)^2$$

The minimizer is obviously $x^\star = 2$. A Pyomo model is three pieces — a `Var`, an `Objective`, and a `solve` call. Pyomo writes the model to an AMPL `.nl` file, POUNCE solves it and writes a `.sol` file, and Pyomo loads the solution back into the model.

In [2]:
m = ConcreteModel()
m.x = Var(initialize=0.5)
m.obj = Objective(expr=(m.x - 2) ** 2)

solver.solve(m)
print(f"x* = {value(m.x):.6f}")

x* = 2.000000


## 2. A constrained problem — HS071

HS071 is the canonical IPOPT test problem:

$$\min_{x \in \mathbb{R}^4} \; x_1 x_4 (x_1 + x_2 + x_3) + x_3$$

$$\text{s.t.} \quad x_1 x_2 x_3 x_4 \geq 25, \quad \sum_i x_i^2 = 40, \quad 1 \leq x_i \leq 5.$$

An indexed `Var` over a `RangeSet` carries the bounds, and the constraints are ordinary algebraic expressions.

In [3]:
m = ConcreteModel()
m.I = RangeSet(1, 4)
m.x = Var(m.I, bounds=(1, 5), initialize={1: 1.0, 2: 5.0, 3: 5.0, 4: 1.0})

m.obj = Objective(
    expr=m.x[1] * m.x[4] * (m.x[1] + m.x[2] + m.x[3]) + m.x[3],
    sense=minimize,
)
m.product = Constraint(expr=m.x[1] * m.x[2] * m.x[3] * m.x[4] >= 25)
m.sphere = Constraint(expr=sum(m.x[i] ** 2 for i in m.I) == 40)

solver.solve(m)

print("x* =", [round(value(m.x[i]), 4) for i in m.I])
print(f"f* = {value(m.obj):.5f}")

x* = [1.0, 4.743, 3.8211, 1.3794]
f* = 17.01402


Expected optimum: $x^\star \approx (1.000,\ 4.743,\ 3.821,\ 1.379)$ with $f^\star \approx 17.014$ — the same point IPOPT returns.

## 3. Dual values

POUNCE writes the constraint Lagrange multipliers into the AMPL `.sol` file next to the primal solution. Pyomo exposes them through a `Suffix` declared with `direction=Suffix.IMPORT`: declare it on the model *before* solving, and Pyomo loads the multipliers back once the solve finishes.

A constraint's dual is its shadow price — the rate at which the optimal objective would change as the constraint is relaxed. Both HS071 constraints are active at the optimum, so both carry a non-zero multiplier. We reuse the model `m` from the previous section.

In [4]:
m.dual = Suffix(direction=Suffix.IMPORT)
solver.solve(m)

print(f"dual[product] = {m.dual[m.product]:.6f}")
print(f"dual[sphere]  = {m.dual[m.sphere]:.6f}")

dual[product] = 0.161469
dual[sphere]  = -0.552294


## 4. Solver options

POUNCE accepts the same `ipopt.opt`-style option keys as IPOPT. Set them on `solver.options` and the plugin forwards them to POUNCE's option list. Common keys are `tol`, `max_iter`, and `print_level`.

In [5]:
solver.options["tol"] = 1e-10
solver.options["max_iter"] = 1000

solver.solve(m)
print(f"f* = {value(m.obj):.10f}")

f* = 17.0140171402


## 5. Watching the solve

Passing `tee=True` to `solve()` streams POUNCE's log into the notebook as the solve runs — the same IPOPT-style iteration table POUNCE prints on the console. It is the quickest way to watch the interior-point iterations drive the primal and dual infeasibilities down to the tolerance.

In [6]:
solver.solve(m, tee=True)

********************************************************************************

                    ####    ###   /   # /#   #/  ####  #####
                    #   #  #   # /#   #/ ##  /  #      #
                    ####   #   #/ #   /  # #/#  #      ####
                    #      #   /  #  /#  # /##  #      #
                    #       ##/    #/#   #/  #   ####  #####

********************************************************************************
This program contains POUNCE, a Rust port of Ipopt for nonlinear optimization.
Released under the Eclipse Public License (EPL) — drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
********************************************************************************

This is POUNCE version 0.3.1-dev, running with linear solver FERAL.

Reading /var/folders/gq/k1kgbl7n539_4dl1md8x3jt80000gn/T/tmpl_3fif56.pyomo.nl...
Parsed 4 vars, 2 cons, jac_nnz=8, h_nnz=10 in 0.00s
Number of nonzeros in equ

{'Problem': [{'Lower bound': -inf, 'Upper bound': inf, 'Number of objectives': 1, 'Number of constraints': 2, 'Number of variables': 4, 'Sense': 'unknown'}], 'Solver': [{'Status': 'ok', 'Message': 'POUNCE 0.3.1-dev\\x3a SolveSucceeded', 'Termination condition': 'optimal', 'Id': 0, 'Error rc': 0, 'Time': 0.006256103515625}], 'Solution': [OrderedDict({'number of solutions': 0, 'number of solutions displayed': 0})]}

## 6. Reading the result object

`solve()` returns a Pyomo `SolverResults`. Its `solver` section carries the status and termination condition POUNCE reported through the SOL file — use these to check that the solve actually converged before trusting the values.

In [7]:
results = solver.solve(m)
print("status:               ", results.solver.status)
print("termination condition:", results.solver.termination_condition)

status:                ok
termination condition: optimal


## 7. Parametric sensitivity (sIPOPT)

Once an NLP is solved you often want to know how the optimum *moves* when a model parameter changes — ideally without paying for a full re-solve. POUNCE implements the **sIPOPT** parametric-sensitivity step: a single linear back-solve against the KKT system at the optimum that predicts the perturbed primal $x^\star + \Delta x$.

It is driven entirely by AMPL suffixes, which Pyomo writes into the `.nl` file. POUNCE auto-detects the request — no special flag or separate binary needed. The recipe:

1. Model each parameter as a `Var` pinned to its nominal value by an equality constraint.
2. Tag the parameter `Var`s with the `sens_state_1` integer suffix (`1, 2, …, n`) and their *perturbed* values with `sens_state_value_1`.
3. Tag the pinning constraints with the matching `sens_init_constr` integer suffix.
4. Declare `sens_sol_state_1` as an `IMPORT` suffix to read the predicted perturbed primal back.

The example below is the classic sIPOPT `parametric` problem:

$$\min_{x}\; x_1^2 + x_2^2 + x_3^2 \quad\text{s.t.}\quad 6x_1 + 3x_2 + 2x_3 = \eta_1,\qquad \eta_2\,x_1 + x_2 - x_3 = 1.$$

We solve at the nominal $(\eta_1, \eta_2) = (5, 1)$ and ask POUNCE to predict the optimum at $\eta_1 \to 4.5$. Every parameter must appear in `sens_state_value_1` — even one whose value is unchanged — because POUNCE computes the perturbation $\Delta p$ against the values it finds there.

In [8]:
m = ConcreteModel()
m.x = Var([1, 2, 3], initialize=1.0)
m.eta1 = Var(initialize=5.0)
m.eta2 = Var(initialize=1.0)

m.obj = Objective(expr=m.x[1] ** 2 + m.x[2] ** 2 + m.x[3] ** 2)
m.con1 = Constraint(expr=6 * m.x[1] + 3 * m.x[2] + 2 * m.x[3] - m.eta1 == 0)
m.con2 = Constraint(expr=m.eta2 * m.x[1] + m.x[2] - m.x[3] - 1 == 0)

# Pin each parameter Var to its nominal value.
m.pin1 = Constraint(expr=m.eta1 == 5.0)
m.pin2 = Constraint(expr=m.eta2 == 1.0)

# Declare the sIPOPT suffixes.
m.sens_state_1 = Suffix(direction=Suffix.EXPORT, datatype=Suffix.INT)
m.sens_state_value_1 = Suffix(direction=Suffix.EXPORT, datatype=Suffix.FLOAT)
m.sens_init_constr = Suffix(direction=Suffix.EXPORT, datatype=Suffix.INT)
m.sens_sol_state_1 = Suffix(direction=Suffix.IMPORT)

# Tag the parameters (1, 2), their perturbed values, and the pinning constraints.
m.sens_state_1[m.eta1] = 1
m.sens_state_1[m.eta2] = 2
m.sens_state_value_1[m.eta1] = 4.5  # perturb eta1: 5 -> 4.5
m.sens_state_value_1[m.eta2] = 1.0  # eta2 unchanged -- but MUST still be listed
m.sens_init_constr[m.pin1] = 1
m.sens_init_constr[m.pin2] = 2

solver.solve(m)

nominal = [round(value(m.x[i]), 6) for i in (1, 2, 3)]
predicted = [round(m.sens_sol_state_1[m.x[i]], 6) for i in (1, 2, 3)]
print("nominal   x* (eta1=5.0) =", nominal)
print("predicted x  (eta1=4.5) =", predicted)

nominal   x* (eta1=5.0) = [0.632653, 0.387755, 0.020408]
predicted x  (eta1=4.5) = [0.576531, 0.377551, -0.045918]


**Verifying the prediction.** This problem has a quadratic objective with *linear* constraints, so the KKT system is constant and the first-order sensitivity step is **exact** — the prediction should match a full re-solve to machine precision. (For a genuinely nonlinear problem the same step gives a first-order approximation.) We check by solving a fresh model with $\eta_1$ pinned at $4.5$.

In [9]:
check = ConcreteModel()
check.x = Var([1, 2, 3], initialize=1.0)
check.eta1 = Var(initialize=4.5)
check.eta2 = Var(initialize=1.0)
check.obj = Objective(expr=check.x[1] ** 2 + check.x[2] ** 2 + check.x[3] ** 2)
check.con1 = Constraint(expr=6 * check.x[1] + 3 * check.x[2] + 2 * check.x[3] - check.eta1 == 0)
check.con2 = Constraint(expr=check.eta2 * check.x[1] + check.x[2] - check.x[3] - 1 == 0)
check.pin1 = Constraint(expr=check.eta1 == 4.5)
check.pin2 = Constraint(expr=check.eta2 == 1.0)

solver.solve(check)

resolved = [round(value(check.x[i]), 6) for i in (1, 2, 3)]
print("re-solved x  (eta1=4.5) =", resolved)
print("max |predicted - re-solved| =", max(abs(p - r) for p, r in zip(predicted, resolved)))

re-solved x  (eta1=4.5) = [0.576531, 0.377551, -0.045918]
max |predicted - re-solved| = 0.0


## Summary

* `import pyomo_pounce` registers `pounce` with `SolverFactory`.
* `SolverFactory('pounce')` is a drop-in replacement for `SolverFactory('ipopt')` — same ASL protocol, same option keys, same result object.
* Primal values, constraint duals (`Suffix.IMPORT`), solver options, and the streamed log (`tee=True`) all work exactly as they do with IPOPT.
* Parametric sensitivity (sIPOPT) is suffix-driven: tag the parameters with `sens_state_1` / `sens_state_value_1` and the pinning constraints with `sens_init_constr`, and POUNCE auto-detects the request and writes the predicted perturbed primal into `sens_sol_state_1`.
* Under the hood Pyomo exchanges AMPL `.nl` / `.sol` files with the `pounce` binary, so no Python extension module is needed — only the binary on `PATH`.

See the [`pyomo-pounce`](https://github.com/jkitchin/pounce/tree/main/pyomo-pounce) directory for the plugin source and installation details.